In [2]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns 


In [3]:
df = pd.read_csv('train.csv')

In [4]:
df.shape

(404290, 6)

In [5]:
df.head()

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [6]:
new_df = df.sample(30000)

In [7]:
new_df.isnull().sum()

id              0
qid1            0
qid2            0
question1       0
question2       1
is_duplicate    0
dtype: int64

In [8]:
new_df = new_df.dropna()

In [9]:
new_df.shape

(29999, 6)

In [10]:
ques_df = new_df[['question1','question2']]
ques_df.head()

,question1,question2
352682,How does the magnet select button for Android ...,Why Cardboard app is not compatible with my Ca...
141061,Which is the best football league in the world?,Which domestic league in world football is the...
7058,What force does the connecting rod handle in t...,Which forces act on connecting rods in Interna...
211725,What are some good ways to detoxify one's body?,Which's the best way to detoxify the body?
111953,How long does it take for an avid weed smoker ...,I smoked weed for 3 days. I was clean before t...


In [11]:
from sklearn.feature_extraction.text import CountVectorizer
# merge texts
questions = list(ques_df['question1']) + list(ques_df['question2'])

cv = CountVectorizer(max_features=3000)
q1_arr, q2_arr = np.vsplit(cv.fit_transform(questions).toarray(),2)

In [12]:
temp_df1 = pd.DataFrame(q1_arr, index= ques_df.index)
temp_df2 = pd.DataFrame(q2_arr, index= ques_df.index)
temp_df = pd.concat([temp_df1, temp_df2], axis=1)
temp_df.shape

(29999, 6000)

In [13]:
temp_df

,0,1,2,3,4,5,6,7,8,9,...,2990,2991,2992,2993,2994,2995,2996,2997,2998,2999
352682,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
141061,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7058,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
211725,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
111953,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209714,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
26451,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
291409,0,0,0,1,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
233531,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
temp_df['is_duplicate'] = new_df['is_duplicate']

In [15]:
temp_df.head()

,0,1,2,3,4,5,6,7,8,9,...,2991,2992,2993,2994,2995,2996,2997,2998,2999,is_duplicate
352682,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
141061,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
7058,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
211725,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
111953,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [16]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(temp_df.iloc[:,0:-1].values,temp_df.iloc[:,-1].values,test_size=0.2,random_state=1)

In [17]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
rf = RandomForestClassifier()
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_test,y_pred)

0.7371666666666666

In [18]:
from xgboost import XGBClassifier
xgb = XGBClassifier()
xgb.fit(X_train,y_train)
y_pred = xgb.predict(X_test)
accuracy_score(y_test,y_pred)

0.7285